In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
df=pd.read_csv('data/dataset.csv')
print(df.head())

                                                text  label
0  Gere faults Trump for blurring meaning of 'ref...      1
1  German parties start to find common ground in ...      1
2  Senate Democratic leader says Attorney General...      1
3  Tennis: Kyrgios fined $10,000 for Shanghai wal...      1
4   Trump Threw Mar-A-Lago Fundraiser For Woman A...      0


In [6]:
df.shape

(45757, 2)

In [7]:
df.columns

Index(['text', 'label'], dtype='object')

In [8]:
df.isnull().sum()   

text     0
label    0
dtype: int64

In [9]:
df.duplicated().sum()

np.int64(0)

In [10]:
import re
import string
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    
    cleaned_text = " ".join(tokens)
    
    return cleaned_text

In [11]:
df.head()

,text,label
0,Gere faults Trump for blurring meaning of 'ref...,1
1,German parties start to find common ground in ...,1
2,Senate Democratic leader says Attorney General...,1
3,"Tennis: Kyrgios fined $10,000 for Shanghai wal...",1
4,Trump Threw Mar-A-Lago Fundraiser For Woman A...,0


In [12]:
df['cleaned_text'] = df['text'].apply(preprocess_text)

In [13]:
df['cleaned_text'].head()

0    gere faults trump blurring meaning refugee ter...
1    german parties start find common ground coalit...
2    senate democratic leader says attorney general...
3    tennis kyrgios fined 10000 shanghai walk tenni...
4    trump threw maralago fundraiser woman center b...
Name: cleaned_text, dtype: object

In [14]:
df.columns

Index(['text', 'label', 'cleaned_text'], dtype='object')

In [15]:
from sklearn.model_selection import train_test_split

X = df['cleaned_text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [16]:
print(X_train.shape)
print(X_test.shape)

(36605,)
(9152,)


In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

X_train_tfidf = tfidf.fit_transform(X_train)    
X_test_tfidf = tfidf.transform(X_test)


In [18]:
print(X_train_tfidf.shape)
print(X_test_tfidf.shape)

(36605, 227975)
(9152, 227975)


Logistic Regression 

In [19]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()
model.fit(X_train_tfidf, y_train)
y_pred = model.predict(X_test_tfidf)
print(y_pred[:10])


[1 0 1 1 0 0 1 0 1 1]


Evaluation 

In [20]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")


Accuracy: 0.9731


In [21]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('model', LogisticRegression(max_iter=1000))
])

scores = cross_val_score(
    pipeline,
    df['cleaned_text'],
    df['label'],
    cv=5,
    scoring='accuracy'
)

print(scores)
print("Mean Accuracy:", scores.mean())
print("Standard Deviation:", scores.std())

[0.97344843 0.97650787 0.97235275 0.97792591 0.97278986]
Mean Accuracy: 0.9746049621616499
Standard Deviation: 0.002207006170892135


In [22]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

conf_matrix = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")  
print(conf_matrix)
print(classification_report(y_test, y_pred))


Confusion Matrix:
[[4463  109]
 [ 137 4443]]
              precision    recall  f1-score   support

           0       0.97      0.98      0.97      4572
           1       0.98      0.97      0.97      4580

    accuracy                           0.97      9152
   macro avg       0.97      0.97      0.97      9152
weighted avg       0.97      0.97      0.97      9152



In [23]:
import pickle

with open("models/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)

In [24]:
with open("models/logistic_model.pkl", "wb") as f:
    pickle.dump(model, f)

Build the Query Generator


User Claim
    ↓
spaCy NER
    ↓
Extract ORG/PERSON/GPE/EVENT
    ↓
Extract Remaining Tokens
    ↓
Remove Stopwords
    ↓
Remove Weak Verbs
    ↓
Merge
    ↓
Final Search Query


Claim
 ↓
Dependency Parse
 ↓
Is there a ccomp?
 ↓
YES                    NO
 ↓                      ↓
Use ccomp verb      Use ROOT verb
as the main event   as the main event
 ↓
Find subject of that event
 ↓
Find object/location/context
 ↓
NER for ORG/PERSON/GPE
 ↓
Build Query

In [ ]:
def extract_event_verb(claim):
    doc = nlp(claim)

    root = None
    ccomp = None

    # Find ROOT
    for token in doc:
        if token.dep_ == "ROOT":
            root = token
            break

    # Find CCOMP attached to ROOT
    if root:
        for child in root.children:
            if child.dep_ == "ccomp":
                ccomp = child
                break

    # Determine actual event verb
    event_verb = ccomp if ccomp else root

    print("Claim:", claim)
    print("ROOT:", root.text if root else None)
    print("CCOMP:", ccomp.text if ccomp else None)
    print("EVENT VERB:", event_verb.text if event_verb else None)
    print("-" * 60)

In [38]:
def extract_event_details(claim):
    doc = nlp(claim)

    root = None
    ccomp = None

    # Find ROOT
    for token in doc:
        if token.dep_ == "ROOT":
            root = token
            break

    # Find CCOMP attached to ROOT
    if root:
        for child in root.children:
            if child.dep_ == "ccomp":
                ccomp = child
                break

    # Determine actual event verb
    event_verb = ccomp if ccomp else root

    subjects = []
    objects = []
    locations = []

    for child in event_verb.children:

        # Subjects
        if child.dep_ in {"nsubj", "nsubjpass"}:
            subjects.append(get_full_phrase(child))

        # Objects
        elif child.dep_ in {"dobj", "obj"}:
            objects.append(get_full_phrase(child))

        # Prepositional objects
        elif child.dep_ == "prep":
            for grandchild in child.children:
                if grandchild.dep_ == "pobj":
                    locations.append(get_full_phrase(grandchild))

    print("Claim:", claim)
    print("EVENT VERB:", event_verb.text)
    print("Subjects:", subjects)
    print("Objects:", objects)
    print("Locations:", locations)
    print("-" * 60)

Above all are test cases... not to implement in final project 

Phase -2 starts from here 

In [44]:
def get_full_phrase(token):
    words = []

    # Get left modifiers
    for child in token.lefts:
        if child.dep_ in {"compound", "amod"}:
            words.extend(get_full_phrase(child).split())

    words.append(token.text)

    return " ".join(words)

In [ ]:
def generate_query(claim):

    doc = nlp(claim)

    query_terms = []

    # -------------------------
    # Find ROOT
    # -------------------------
    root = None

    for token in doc:
        if token.dep_ == "ROOT":
            root = token
            break

    # -------------------------
    # Find CCOMP
    # -------------------------
    ccomp = None

    if root:
        for child in root.children:
            if child.dep_ == "ccomp":
                ccomp = child
                break

    # Actual event verb
    event_verb = ccomp if ccomp else root

    # -------------------------
    # NER Entities
    # -------------------------
    for ent in doc.ents:
        if ent.label_ in {"PERSON", "ORG", "GPE", "LOC"}:
            query_terms.append(ent.text)

    # -------------------------
    # Event Verb
    # -------------------------
    if event_verb:
        query_terms.append(event_verb.text)

    # -------------------------
    # Explore Event Verb
    # -------------------------
    if event_verb:

        for child in event_verb.children:

            # Subjects
            if child.dep_ in {"nsubj", "nsubjpass"}:
                query_terms.append(get_full_phrase(child))

            # Objects
            elif child.dep_ in {"dobj", "obj"}:

                query_terms.append(
                    get_full_phrase(child)
                )

                # Explore object children
                for grandchild in child.children:

                    # Locations/context
                    if grandchild.dep_ == "prep":

                        for ggc in grandchild.children:

                            if ggc.dep_ == "pobj":

                                query_terms.append(
                                    get_full_phrase(ggc)
                                )

            # Verb-level locations/context
            elif child.dep_ == "prep":

                for grandchild in child.children:

                    if grandchild.dep_ == "pobj":

                        query_terms.append(
                            get_full_phrase(grandchild)
                        )

    # -------------------------
    # Remove duplicates
    # -------------------------
    final_terms = []
    seen = set()

    for term in query_terms:

        key = term.lower()

        if key not in seen:
            final_terms.append(term)
            seen.add(key)

    return " ".join(final_terms)

Now we implement the Retrieval Engine by using News API

In [25]:
import requests

API_KEY = "12e6234009694ace8e2c1f423a33ad07"

In [48]:
def retrieve_articles(query, page_size=10):

    url = "https://newsapi.org/v2/everything"

    params = {
        "q": query,
        "apiKey": API_KEY,
        "language": "en",
        "sortBy": "relevancy",
        "pageSize": page_size
    }

    response = requests.get(url, params=params)

    # Check if request succeeded
    if response.status_code != 200:
        print("Error:", response.status_code)
        print(response.json())
        return []

    data = response.json()

    articles = []

    for article in data.get("articles", []):

        articles.append({
            "title": article.get("title"),
            "description": article.get("description"),
            "source": article.get("source", {}).get("name"),
            "publishedAt": article.get("publishedAt"),
            "url": article.get("url")
        })

    return articles

Sentence-BERT Similarity checking 

In [3]:
from sentence_transformers import SentenceTransformer

similarity_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

c:\Users\Aravind M\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Aravind M\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Aravind M\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer M

In [6]:
from transformers import pipeline

nli = pipeline(
    "text-classification",
    model="MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli",
    device=-1   # CPU
)

c:\Users\Aravind M\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Aravind M\.cache\huggingface\hub\models--MoritzLaurer--DeBERTa-v3-base-mnli-fever-anli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 202/202 [00:00<00:00, 534.2

In [15]:
import joblib

loaded_vectorizer = joblib.load("models/tfidf_vectorizer.pkl")
loaded_model = joblib.load("models/logistic_model.pkl")

In [17]:
def predict_fake_news(claim):
    
    vectorized_claim = loaded_vectorizer.transform([claim])

    prediction = loaded_model.predict(vectorized_claim)[0]

    probability = loaded_model.predict_proba(vectorized_claim)[0]

    fake_probability = probability[1]

    return {
        "prediction": "Fake" if prediction == 1 else "Real",
        "fake_probability": fake_probability
    }

In [27]:
import requests
import spacy
import joblib
from sentence_transformers import util

In [29]:
nlp = spacy.load("en_core_web_sm")

In [31]:
loaded_vectorizer = joblib.load("models/tfidf_vectorizer.pkl")
loaded_model = joblib.load("models/logistic_model.pkl")

In [32]:
def predict_fake_news(claim):

    x = loaded_vectorizer.transform([claim])

    pred = loaded_model.predict(x)[0]

    probs = loaded_model.predict_proba(x)[0]

    fake_prob = probs[1]

    return {
        "prediction": "Fake" if pred == 1 else "Real",
        "fake_probability": float(fake_prob)
    }

In [33]:
def generate_query(claim):

    doc = nlp(claim)

    entities = []

    for ent in doc.ents:
        if ent.label_ in {"PERSON", "ORG", "GPE", "LOC"}:
            entities.append(ent.text)

    if len(entities) > 0:
        return " ".join(entities)

    return claim

In [34]:
def retrieve_articles(query, page_size=10):

    url = "https://newsapi.org/v2/everything"

    params = {
        "q": query,
        "apiKey": API_KEY,
        "language": "en",
        "sortBy": "relevancy",
        "pageSize": page_size
    }

    response = requests.get(url, params=params)

    if response.status_code != 200:
        return []

    data = response.json()

    articles = []

    for article in data.get("articles", []):

        articles.append({
            "title": article.get("title"),
            "description": article.get("description"),
            "source": article.get("source", {}).get("name"),
            "publishedAt": article.get("publishedAt"),
            "url": article.get("url")
        })

    return articles

In [35]:
def rank_articles(claim, articles, top_k=3):

    claim_embedding = similarity_model.encode(
        claim,
        convert_to_tensor=True
    )

    scored = []

    for article in articles:

        text = (
            (article["title"] or "")
            + " "
            + (article["description"] or "")
        )

        article_embedding = similarity_model.encode(
            text,
            convert_to_tensor=True
        )

        score = util.cos_sim(
            claim_embedding,
            article_embedding
        ).item()

        article["similarity"] = score

        scored.append(article)

    scored.sort(
        key=lambda x: x["similarity"],
        reverse=True
    )

    return scored[:top_k]

In [43]:
def verify_evidence(claim, evidence_text):

    result = nli({
        "text": evidence_text,
        "text_pair": claim
    })

    return {
        "label": result["label"].upper(),
        "score": float(result["score"])
    }

In [40]:
def verify_news(claim):

    report = {}

    # Phase 1
    historical = predict_fake_news(claim)

    report["historical_prediction"] = historical

    # Retrieval
    query = generate_query(claim)

    report["query"] = query

    articles = retrieve_articles(query)

    report["articles_found"] = len(articles)

    if len(articles) == 0:

        report["evidence_verdict"] = "INSUFFICIENT EVIDENCE"

        return report

    # Ranking
    top_articles = rank_articles(
        claim,
        articles,
        top_k=3
    )

    report["top_evidence"] = []

    final_verdict = "INSUFFICIENT EVIDENCE"

    for article in top_articles:

        evidence_text = (
            (article["title"] or "")
            + ". "
            + (article["description"] or "")
        )

        nli_result = verify_evidence(
            claim,
            evidence_text
        )

        report["top_evidence"].append({
            "title": article["title"],
            "source": article["source"],
            "similarity": article["similarity"],
            "nli_label": nli_result["label"],
            "nli_score": nli_result["score"],
            "url": article["url"]
        })

        if (
            nli_result["label"] == "CONTRADICTION"
            and nli_result["score"] >= 0.90
        ):
            final_verdict = "REFUTED"

        elif (
            nli_result["label"] == "ENTAILMENT"
            and nli_result["score"] >= 0.90
            and final_verdict != "REFUTED"
        ):
            final_verdict = "SUPPORTED"

    report["evidence_verdict"] = final_verdict

    return report